# PPT Screenshot Notebook — Data + Outputs + Analysis (No File Saving)

This notebook is designed for **screenshots**. It reads the latest exported CSV outputs and produces clean tables/figures **inline only** (no saving to `outputs/`).

Recommended run: use the latest *3D + XGBoost* comparison outputs: **tag = 083, mode = live**.

You can change `TAG` below to `02` if you want to use the earlier run without XGBoost.


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

ROOT = Path(r'C:/Users/user/Downloads/Moltbot/HKU-FYP/fyp_finance_ml_v2')
OUT = ROOT / 'outputs'

TAG = '083'
MODE = 'live'

metrics_path = OUT / 'metrics' / f'{TAG}_{MODE}_metrics.csv'
bt_path = OUT / 'metrics' / f'{TAG}_{MODE}_backtest_summary.csv'
rel_path = OUT / 'metrics' / f'{TAG}_{MODE}_relative_summary.csv'
bucket_path = OUT / 'tables' / f'{TAG}_{MODE}_bucket_returns.csv'
dq_path = OUT / 'tables' / f'{TAG}_{MODE}_data_quality.csv'
curves_dir = OUT / 'curves'

assert metrics_path.exists(), metrics_path
assert bt_path.exists(), bt_path
assert rel_path.exists(), rel_path
print('Loaded paths OK')

## 1) Present the data (input summary)
We use `data_quality.csv` as the reportable summary of the downloaded dataset coverage.

In [ ]:
dq = pd.read_csv(dq_path)
dq.head()

In [ ]:
# Screenshot-friendly dataset coverage table
dq_summary = dq.agg({
    'ticker': 'nunique',
    'row_count': ['sum','mean','min','max'],
    'close_na_ratio': ['mean','max'],
})
dq_summary

## 2) Present the outputs (what we produce)
Three-tier evaluation:
- **ML metrics** (accuracy / balanced accuracy)
- **Signal metrics** (Rank IC, top-k hit rate, spreads)
- **Portfolio performance** (Sharpe, max drawdown, returns)

In [ ]:
metrics = pd.read_csv(metrics_path)
bt = pd.read_csv(bt_path)
rel = pd.read_csv(rel_path)

print('metrics rows:', len(metrics))
print('backtest rows:', len(bt))
print('relative rows:', len(rel))

metrics[['feature_set','model','horizon_days','balanced_accuracy','rank_ic','top_k_hit_rate']].head()

### 2.1 One clean backtest summary table (3D focus)
Pick one execution mode for the slide (recommended: **close_to_close**), but you can also show both.

In [ ]:
EXEC_MODE = 'close_to_close'
H = 3

bt3 = bt[(bt.horizon_days==H) & (bt.execution_mode==EXEC_MODE)].copy()
rel3 = rel[(rel.horizon_days==H) & (rel.execution_mode==EXEC_MODE)].copy()

summary = (bt3.merge(rel3, on=['feature_set','model','horizon_days','execution_mode'], how='left')
             .merge(metrics[metrics.horizon_days==H], on=['feature_set','model','horizon_days'], how='left'))

cols = ['feature_set','model','execution_mode','sharpe','annualized_return','max_drawdown','avg_turnover','avg_cost_drag',
        'alpha_ann','information_ratio','balanced_accuracy','rank_ic']
summary_table = summary[cols].sort_values('sharpe', ascending=False)
summary_table

## 3) Results analysis: classification (balanced accuracy)
Goal: show that predictive edge is small (around 0.50–0.51).

In [ ]:
# Balanced accuracy bar chart (3D only)
m3 = metrics[metrics.horizon_days==H].copy()
m3['label'] = m3['feature_set'] + ' | ' + m3['model']

plt.figure(figsize=(12,4))
order = m3.sort_values('balanced_accuracy', ascending=False)
plt.bar(order['label'], order['balanced_accuracy'])
plt.axhline(0.5, color='black', linewidth=1)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.title('Balanced Accuracy (3D) — small edge around 0.50')
plt.ylabel('Balanced Accuracy')
plt.tight_layout()
plt.show()

## 4) Signal analysis: Rank IC / top-k hit rate
Goal: show whether any feature set is consistently better.

In [ ]:
# Rank IC bar chart (3D)
plt.figure(figsize=(12,4))
order = m3.sort_values('rank_ic', ascending=False)
plt.bar(order['label'], order['rank_ic'])
plt.axhline(0.0, color='black', linewidth=1)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.title('Rank IC (3D) — generally small / unstable')
plt.ylabel('Rank IC (Spearman)')
plt.tight_layout()
plt.show()

# Top-k hit rate
plt.figure(figsize=(12,4))
order = m3.sort_values('top_k_hit_rate', ascending=False)
plt.bar(order['label'], order['top_k_hit_rate'])
plt.axhline(0.5, color='black', linewidth=1)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.title('Top-K Hit Rate (3D) — fraction of top-K with positive forward return')
plt.ylabel('Hit Rate')
plt.tight_layout()
plt.show()

## 5) Historical performance (equity curves)
Goal: show F1 + RF + 3D wins, and more features ≠ better.

We plot SPY plus a few selected strategies under the chosen execution mode.

In [ ]:
def load_curve(path: Path):
    d = pd.read_csv(path)
    d['date'] = pd.to_datetime(d['date'])
    d = d.sort_values('date')
    # NOTE: do NOT compute equity here; we will align date ranges first.
    return d

def curve_file(fs, model, h, em):
    return curves_dir / f'{TAG}_{MODE}_{fs}_{model}_{int(h)}d_{em}_daily.csv'

def align_and_rebase(curves: dict):
    """Align all curves to a common overlapping date range and rebase equity to 1 at the common start.

    This fixes the PPT issue where SPY starts earlier (e.g., 2016) than strategy curves.
    """
    # compute common overlap
    starts = [df['date'].min() for df in curves.values()]
    ends = [df['date'].max() for df in curves.values()]
    start = max(starts)
    end = min(ends)
    out = {}
    for name, df in curves.items():
        w = df[(df['date'] >= start) & (df['date'] <= end)].copy()
        w = w.sort_values('date')
        w['equity'] = (1 + w['net_ret'].astype(float)).cumprod()
        # rebase to 1 exactly (in case first row has missing)
        if len(w) > 0:
            w['equity'] = w['equity'] / float(w['equity'].iloc[0])
        out[name] = w
    return out, start, end

# Choose 4 strategies to compare (edit these if you want)
choices = [
    ('F1_momentum','random_forest'),
    ('F2_momentum_reversal','random_forest'),
    ('F3_plus_risk_liquidity','random_forest'),
    ('F5_full_finance_no_fundamental','xgboost'),
]

spy_path = curves_dir / f'{TAG}_{MODE}_SPY_{H}d_daily.csv'
curves = {}

# Load SPY (benchmark)
if spy_path.exists():
    curves['SPY'] = load_curve(spy_path)
else:
    print('Missing SPY curve file:', spy_path)

# Load strategies
for fs, m in choices:
    p = curve_file(fs, m, H, EXEC_MODE)
    name = f'{fs}|{m}'
    if p.exists():
        curves[name] = load_curve(p)
    else:
        print('Missing curve file:', p)

# Align to common period and rebase equity
curves_aligned, start, end = align_and_rebase(curves)
print('Common plot range:', start.date(), 'to', end.date())

plt.figure(figsize=(12,4))
for name, df in curves_aligned.items():
    lw = 2 if name=='SPY' else 1.5
    plt.plot(df['date'], df['equity'], label=name, linewidth=lw)

plt.title(f'Equity Curves (h={H}D, {EXEC_MODE}) — aligned start/end')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6) Strategy landscape: Sharpe by feature set (close-to-close vs open-to-open)
Goal: show "more features ≠ better" and compare execution conventions.

In [ ]:
def sharpe_bar(bt, h, em, title):
    sub = bt[(bt.horizon_days==h) & (bt.execution_mode==em)].copy()
    sub['label'] = sub['feature_set'] + ' | ' + sub['model']
    sub = sub.sort_values('sharpe', ascending=False)
    plt.figure(figsize=(12,4))
    plt.bar(sub['label'], sub['sharpe'])
    plt.axhline(0.0, color='black', linewidth=1)
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.title(title)
    plt.ylabel('Sharpe')
    plt.tight_layout()
    plt.show()

sharpe_bar(bt, H, 'close_to_close', f'Sharpe (3D) — close_to_close')
sharpe_bar(bt, H, 'open_to_open', f'Sharpe (3D) — open_to_open')

## 7) Optional: Rolling Sharpe (regime dependence)
This shows whether strategies win in some periods and lose in others.

In [ ]:
def rolling_sharpe(rets, window=63, periods_per_year=252):
    r = pd.Series(rets).astype(float)
    mu = r.rolling(window).mean() * periods_per_year
    vol = r.rolling(window).std(ddof=0) * np.sqrt(periods_per_year)
    return mu / vol

WINDOW = 63

# Rolling Sharpe for winner vs SPY (aligned to the same date range)
winner_fs, winner_model = 'F1_momentum','random_forest'
p = curve_file(winner_fs, winner_model, H, EXEC_MODE)

if p.exists() and spy_path.exists():
    strat = load_curve(p)
    spy = load_curve(spy_path)

    # align date ranges first (so SPY doesn't start earlier)
    tmp_curves, start, end = align_and_rebase({'Strategy': strat, 'SPY': spy})
    strat_a = tmp_curves['Strategy'].set_index('date')
    spy_a = tmp_curves['SPY'].set_index('date')

    rs = rolling_sharpe(strat_a['net_ret'], WINDOW).rename('Strategy')
    rb = rolling_sharpe(spy_a['net_ret'], WINDOW).rename('SPY')
    tmp = pd.concat([rs, rb], axis=1)

    plt.figure(figsize=(12,4))
    plt.plot(tmp.index, tmp['Strategy'], label=f'{winner_fs}|{winner_model}')
    plt.plot(tmp.index, tmp['SPY'], label='SPY')
    plt.axhline(0.0, color='black', linewidth=1)
    plt.title(f'Rolling Sharpe (window={WINDOW}D, h=3D, {EXEC_MODE}) — aligned')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Missing curve files for rolling sharpe plot.')